# Cambrian-P results report: charts and slide deck

Reads everything the Cambrian-P notebooks saved in your Drive `results` folder:
- your own videos, answered by the five Cambrian-P models and Cambrian-S-7B;
- the VSI-Bench run in `results/benchmark`.

It draws charts and camera-path maps, saves them as PNGs, and builds a slide deck (`.pptx`, opens in Google Slides) with one chart per slide.

**No GPU needed.** A free CPU runtime is enough. Run all cells; it takes a minute or two.

**Output**, in `REPORT_DIR` on your Drive:
- `cambrian_p_results.pptx`: the deck. In Drive, right-click it → Open with → Google Slides.
- `charts/`: every chart as a PNG.
- `own_videos_scored.csv` and `benchmark_scored.csv`: every answer with its score.

In [ ]:
# @title 1. Settings and Google Drive
RESULTS_DIR = "/content/drive/MyDrive/Cambrian-P/results"  # @param {type:"string"}
REPORT_DIR = "/content/drive/MyDrive/Cambrian-P/results/report"  # @param {type:"string"}
DECK_TITLE = "Cambrian-P: results on our videos and VSI-Bench"  # @param {type:"string"}

# Models in chart order, with the frame count each was run with (None = not a Cambrian-P model).
MODELS = {
    "Cambrian-P-7B": 128,
    "Cambrian-P-7B-32f": 32,
    "Cambrian-P-7B-Mix-MA": 64,
    "Cambrian-P-7B-Mix-3R": 32,
    "Cambrian-P-7B-Mix-CamS": 128,
    "Cambrian-S-7B": None,
}
BASELINE = "Cambrian-S-7B"   # the model Cambrian-P is fine-tuned from, without camera tokens
MAIN = "Cambrian-P-7B"

import os, subprocess, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-pptx"], check=True)
except ImportError:
    pass  # not in Colab: RESULTS_DIR must be a local folder and python-pptx installed
BENCH_DIR = os.path.join(RESULTS_DIR, "benchmark")
CHART_DIR = os.path.join(REPORT_DIR, "charts")
os.makedirs(CHART_DIR, exist_ok=True)
own_models = [m for m in MODELS if os.path.isdir(os.path.join(RESULTS_DIR, m))]
bench_models = [m for m in MODELS if os.path.isdir(os.path.join(BENCH_DIR, m))]
print("Your videos:", ", ".join(own_models) or "no model folders found")
print("VSI-Bench:  ", ", ".join(bench_models) or "no benchmark folder found")

In [ ]:
# @title 2. Load and score every answer
import csv, glob, json, re
import numpy as np
import pandas as pd

def first_word(answer):
    return answer.split(" ")[0].rstrip(".").strip()

def as_float(text):
    try:
        return float(text)
    except ValueError:
        return None

def score(answer, truth):
    """VSI-Bench scoring (lmms-eval/lmms_eval/tasks/vsibench/utils.py): exact match on the first word for
    letters and words, mean relative accuracy for numbers; NaN when there is no true answer."""
    truth = str(truth).strip()
    if not truth:
        return np.nan
    if as_float(truth) is not None:
        target, pred = as_float(truth), as_float(first_word(answer))
        if pred is None:
            return 0.0
        if target == 0:
            return float(pred == 0)
        return float(np.mean(abs(pred - target) / abs(target) <= 1 - np.linspace(0.5, 0.95, 10)))
    if len(truth.split()) == 1:
        return float(first_word(answer).lower() == truth.lower())
    return float(answer.strip().rstrip(".").lower() == truth.rstrip(".").lower())

FRAMING = ("These are frames of a video.", "Please answer the question", "Answer with the option", "Options:")

def question_text(question):
    """The question sentence alone, without the VSI-Bench framing or the lettered options."""
    lines = [l for l in question.splitlines() if l.strip() and not l.startswith(FRAMING) and not re.match(r"^[A-H]\. ", l)]
    return " ".join(lines)

def vsi_type(question):
    """VSI-Bench question type, recognised from the question wording."""
    q = question.lower()
    rules = [("you are a robot", "Route Plan"), ("first-time appearance order", "Appr. Order"),
             ("standing by", "Rel. Dir."), ("which of these objects", "Rel. Dist."),
             ("longest dimension", "Obj. Size"), ("size of this room", "Room Size"),
             ("how many", "Obj. Count"), ("distance between", "Abs. Dist.")]
    return next((t for key, t in rules if key in q), "Other")

def load_runs(base, models):
    """Rows from each model's results.csv, or from per-video answers.json if a run stopped before writing it."""
    rows = []
    for m in models:
        path = os.path.join(base, m, "results.csv")
        if os.path.exists(path):
            with open(path, newline="") as f:
                got = list(csv.DictReader(f))
        else:
            got = [r for p in sorted(glob.glob(os.path.join(base, m, "*", "answers.json"))) for r in json.load(open(p))]
        rows += [dict(r, model=m) for r in got if not r.get("model_answer", "").startswith("ERROR:")]
    df = pd.DataFrame(rows, columns=["video", "question", "true_answer", "model_answer", "model"]).fillna("")
    df["score"] = [score(a, t) for a, t in zip(df["model_answer"], df["true_answer"])]
    df["q"] = df["question"].map(question_text)
    order = df.drop_duplicates(["video", "question"])[["video", "question"]].reset_index(drop=True)
    order["qid"] = order.groupby("video").cumcount() + 1
    return df.merge(order, on=["video", "question"])

own = load_runs(RESULTS_DIR, own_models)
own_videos = list(dict.fromkeys(own["video"]))
own_scored = own.dropna(subset=["score"])
print(f"Your videos: {len(own_models)} models, {len(own_videos)} videos, "
      f"{own[['video', 'question']].drop_duplicates().shape[0]} questions "
      f"({own_scored[['video', 'question']].drop_duplicates().shape[0]} with a true answer)")

bench = load_runs(BENCH_DIR, bench_models)
bench["type"] = bench["question"].map(vsi_type)
bench_videos = list(dict.fromkeys(bench["video"]))
print(f"VSI-Bench: {len(bench_models)} models, {len(bench_videos)} scenes, "
      f"{bench[['video', 'question']].drop_duplicates().shape[0]} questions")

# The benchmark summary table written alongside the benchmark run (per-type scores, 95% range, paper numbers).
bench_table = None
p = os.path.join(BENCH_DIR, "benchmark_scores.csv")
if os.path.exists(p):
    bench_table = pd.read_csv(p, index_col=0)
    bench_table.columns = [c.strip() for c in bench_table.columns]
    print("Loaded benchmark_scores.csv:", ", ".join(bench_table.index))
    # Check the question types recognised here against the counts in that table.
    if len(bench):
        counts = bench.drop_duplicates(["video", "question"])["type"].value_counts()
        for c in bench_table.columns:
            m = re.match(r"(.+) \(n=(\d+)\)", c)
            if m and m.group(1) != "Overall" and counts.get(m.group(1), 0) != int(m.group(2)):
                print(f"  Note: {m.group(1)}: {counts.get(m.group(1), 0)} questions recognised here, table says {m.group(2)}")

own[["video", "qid", "q", "true_answer", "model", "model_answer", "score"]].rename(columns={"q": "question"}) \
    .to_csv(os.path.join(REPORT_DIR, "own_videos_scored.csv"), index=False)
bench[["video", "type", "q", "true_answer", "model", "model_answer", "score"]].rename(columns={"q": "question"}) \
    .to_csv(os.path.join(REPORT_DIR, "benchmark_scored.csv"), index=False)

In [ ]:
# @title 3. Chart style
import textwrap
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import Image, display

SURFACE, INK, INK_2, GRID, NEUTRAL = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df", "#f0efec"
BLUE, ORANGE = "#2a78d6", "#eb6834"                       # categorical slots 1 and 2
GOOD, PARTIAL, BAD = "#0ca30c", "#ec835a", "#d03b3b"      # status colors: always paired with ✓ ~ ✗
GAIN, LOSS = "#2a78d6", "#e34948"                         # diverging poles
BLUES = LinearSegmentedColormap.from_list("blues", ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])
mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": INK_2, "xtick.color": INK_2, "ytick.color": INK_2,
    "axes.edgecolor": GRID, "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.spines.top": False, "axes.spines.right": False,
})

def label(m, newline=False):
    short = m.replace("Cambrian-", "")
    f = MODELS.get(m)
    tag = f"{f}f" if f else "base, no camera tokens" if m == BASELINE else ""
    return f"{short}{chr(10) if newline else ' '}({tag})" if tag else short

def ink_on(rgba):
    r, g, b = rgba[:3]
    return INK if 0.2126 * r + 0.7152 * g + 0.0722 * b > 0.55 else "white"

def clean(ax):
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)

CHARTS = []   # (file, slide title, footnote), in deck order

def save(fig, name, title, note=""):
    path = os.path.join(CHART_DIR, name)
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)
    CHARTS.append((path, title, note))
    display(Image(path, width=900))

def heatmap(ax, values, rows, cols, fmt, vmax=100, empty="—"):
    im = ax.imshow(np.asarray(values, dtype=float), cmap=BLUES, vmin=0, vmax=vmax, aspect="auto")
    for i in range(len(rows)):
        for j in range(len(cols)):
            v = values[i][j]
            if v is None or np.isnan(v):
                ax.text(j, i, empty, ha="center", va="center", color=INK_2)
            else:
                ax.text(j, i, fmt(v), ha="center", va="center", color=ink_on(im.cmap(im.norm(v))), fontweight="bold")
    ax.set_xticks(range(len(cols)), cols)
    ax.set_yticks(range(len(rows)), rows)
    ax.set_xticks(np.arange(len(cols) + 1) - 0.5, minor=True)
    ax.set_yticks(np.arange(len(rows) + 1) - 0.5, minor=True)
    ax.grid(which="minor", color=SURFACE, lw=2)
    ax.tick_params(which="minor", length=0)
    clean(ax)
    return im

## VSI-Bench (ScanNet scenes)

In [ ]:
# @title 4. VSI-Bench charts
TYPES = ["Obj. Count", "Abs. Dist.", "Obj. Size", "Room Size", "Rel. Dist.", "Rel. Dir.", "Route Plan", "Appr. Order"]

def col(name):
    return next((c for c in bench_table.columns if c.startswith(name + " (") or c == name), None)

if bench_table is not None:
    runs = [m for m in MODELS if m in bench_table.index]
    overall_col = col("Overall")
    n_total = re.search(r"n=(\d+)", overall_col).group(1) if overall_col and "n=" in overall_col else "?"
    ci = bench_table.get("Overall 95% range")
    # Chart: overall score per model, with 95% range, paper numbers and chance level.
    fig, ax = plt.subplots(figsize=(10, 0.62 * len(runs) + 1.6))
    y = np.arange(len(runs))[::-1]
    vals = bench_table.loc[runs, overall_col].astype(float)
    ax.barh(y, vals, height=0.56, color=BLUE, label="Our run")
    for yi, m in zip(y, runs):
        lo_hi = str(ci.get(m, "")) if ci is not None else ""
        parts = re.split(r"\s*[–-]\s*", lo_hi) if lo_hi and lo_hi != "nan" else []
        if len(parts) == 2:
            lo, hi = map(float, parts)
            ax.plot([lo, hi], [yi, yi], color=INK, lw=1.5)
            ax.plot([lo, lo], [yi - 0.12, yi + 0.12], color=INK, lw=1.5)
            ax.plot([hi, hi], [yi - 0.12, yi + 0.12], color=INK, lw=1.5)
        paper = f"{m} (paper)"
        if paper in bench_table.index and not np.isnan(bench_table.loc[paper, overall_col]):
            ax.plot(bench_table.loc[paper, overall_col], yi, marker="D", ms=9, color=ORANGE, mec=SURFACE, mew=1.5,
                    label="Paper" if "Paper" not in [h.get_label() for h in ax.lines] else None)
        ax.text(max(vals[m], float(parts[1]) if len(parts) == 2 else 0) + 1.2, yi, f"{vals[m]:.1f}%", va="center")
    chance = next((r for r in bench_table.index if r.lower().startswith("chance")), None)
    if chance:
        cv = float(bench_table.loc[chance, overall_col])
        ax.axvline(cv, color=INK_2, ls="--", lw=1)
        ax.text(cv, y[0] + 0.55, f" chance {cv:.0f}%", color=INK_2, fontsize=9, va="bottom")
    ax.set_yticks(y, [label(m) for m in runs])
    ax.set_xlim(0, 100)
    ax.set_xlabel(f"VSI-Bench score, % ({n_total} questions; whiskers = 95% range of our run)")
    ax.xaxis.grid(True, color=GRID)
    ax.set_axisbelow(True)
    clean(ax)
    ax.legend(loc="lower right", frameon=False)
    ax.set_title("VSI-Bench overall score")
    best = vals.idxmax()
    gain = vals.get(MAIN, np.nan) - vals.get(BASELINE, np.nan)
    title = (f"{label(MAIN)} scores {vals[MAIN]:.1f}% on VSI-Bench, {gain:+.1f} points over {BASELINE}"
             if MAIN in vals and BASELINE in vals else f"Best VSI-Bench score: {label(best)} ({vals[best]:.1f}%)")
    save(fig, "bench_1_overall.png", title,
         "ScanNet scenes from VSI-Bench. Diamonds: the paper's full-benchmark numbers. Dashed line: answering with the most frequent answer.")

    # Chart: score by question type, our runs and the paper's.
    cols = [col(t) for t in TYPES if col(t)] + ([overall_col] if overall_col else [])
    rows = runs + [r for r in bench_table.index if "(paper)" in r] + ([chance] if chance else [])
    values = bench_table.loc[rows, cols].astype(float).values.tolist()
    fig, ax = plt.subplots(figsize=(1.25 * len(cols) + 3.5, 0.55 * len(rows) + 1.6))
    heatmap(ax, values, [label(r) if r in MODELS else r for r in rows], [c.replace(" (", "\n(") for c in cols], lambda v: f"{v:.0f}")
    ax.axhline(len(runs) - 0.5, color=INK, lw=2)
    ax.xaxis.tick_top()
    ax.set_title("VSI-Bench score by question type (%)", pad=44)
    by_type = bench_table.loc[MAIN, [col(t) for t in TYPES if col(t)]].astype(float) if MAIN in bench_table.index else None
    title = (f"{label(MAIN)} is strongest on {by_type.idxmax().split(' (')[0]} ({by_type.max():.0f}%) "
             f"and weakest on {by_type.idxmin().split(' (')[0]} ({by_type.min():.0f}%)") if by_type is not None else "Score by question type"
    save(fig, "bench_2_by_type.png", title, "Rows above the line: our runs. Below: the paper's numbers and chance level. n = questions of that type.")

    # Chart: what the camera tokens add, by question type (Cambrian-P-7B minus Cambrian-S-7B).
    if MAIN in bench_table.index and BASELINE in bench_table.index:
        tcols = [col(t) for t in TYPES if col(t)] + [overall_col]
        ours = (bench_table.loc[MAIN, tcols].astype(float) - bench_table.loc[BASELINE, tcols].astype(float))
        pp, ps = f"{MAIN} (paper)", f"{BASELINE} (paper)"
        paper_gain = (bench_table.loc[pp, tcols].astype(float) - bench_table.loc[ps, tcols].astype(float)) \
            if pp in bench_table.index and ps in bench_table.index else None
        fig, ax = plt.subplots(figsize=(10, 0.55 * len(tcols) + 1.4))
        y = np.arange(len(tcols))[::-1]
        ax.barh(y, ours.values, height=0.56, color=[GAIN if v >= 0 else LOSS for v in ours.values], label="Our run")
        if paper_gain is not None:
            ax.scatter(paper_gain.values, y, marker="D", s=60, color=ORANGE, edgecolor=SURFACE, lw=1.5, zorder=3, label="Paper")
        for yi, v in zip(y, ours.values):
            ax.text(v + (0.8 if v >= 0 else -0.8), yi, f"{v:+.1f}", va="center", ha="left" if v >= 0 else "right")
        ax.axvline(0, color=INK, lw=1)
        ax.set_yticks(y, [c.split(" (")[0] for c in tcols])
        span = max(abs(ours).max(), abs(paper_gain).max() if paper_gain is not None else 0) + 6
        ax.set_xlim(-span, span)
        ax.set_xlabel(f"Points gained by {MAIN} over {BASELINE} (blue = better, red = worse)")
        ax.xaxis.grid(True, color=GRID)
        ax.set_axisbelow(True)
        clean(ax)
        ax.legend(loc="upper left", frameon=False)
        ax.set_title("What Cambrian-P's camera tokens add, by question type")
        top = ours.drop(overall_col).idxmax()
        save(fig, "bench_3_gain.png",
             f"Camera tokens help most on {top.split(' (')[0]} ({ours[top]:+.1f} points); overall {ours[overall_col]:+.1f}",
             "Same scenes and questions for both models; the paper's gain is over the full benchmark.")

if len(bench):
    # Chart: score per scene and model, from the per-question answers.
    per_scene = bench.dropna(subset=["score"]).pivot_table(index="model", columns="video", values="score", aggfunc="mean")
    per_scene = per_scene.reindex([m for m in MODELS if m in per_scene.index])
    scenes = list(per_scene.columns)
    nq = bench.drop_duplicates(["video", "question"]).groupby("video").size()
    fig, ax = plt.subplots(figsize=(1.05 * len(scenes) + 3.5, 0.6 * len(per_scene) + 1.8))
    heatmap(ax, (100 * per_scene.values).tolist(), [label(m) for m in per_scene.index],
            [f"{s.replace('scannet__', '').replace('.mp4', '')}\n({nq.get(s, 0)} q)" for s in scenes], lambda v: f"{v:.0f}")
    plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
    ax.set_title("VSI-Bench score per scene (%)")
    hard = per_scene.mean().idxmin()
    save(fig, "bench_4_per_scene.png",
         f"Hardest scene for every model on average: {hard.replace('scannet__', '').replace('.mp4', '')} "
         f"({100 * per_scene.mean()[hard]:.0f}%)",
         "Mean score over each scene's questions (VSI-Bench scoring, computed here from the saved answers).")

## Our videos

In [ ]:
# @title 5. Charts for our own videos
if len(own_scored):
    # Overall score per model.
    summ = own_scored.groupby("model")["score"].agg(["mean", "sum", "count"]).reindex([m for m in MODELS if m in own_models])
    fig, ax = plt.subplots(figsize=(10, 0.62 * len(summ) + 1.4))
    y = np.arange(len(summ))[::-1]
    ax.barh(y, 100 * summ["mean"], height=0.56, color=BLUE)
    for yi, (m, r) in zip(y, summ.iterrows()):
        ax.text(100 * r["mean"] + 1.2, yi, f"{100 * r['mean']:.0f}%  ({r['sum']:g}/{r['count']:.0f})", va="center")
    ax.set_yticks(y, [label(m) for m in summ.index])
    ax.set_xlim(0, 110)
    ax.set_xticks([0, 25, 50, 75, 100], ["0%", "25%", "50%", "75%", "100%"])
    ax.xaxis.grid(True, color=GRID)
    ax.set_axisbelow(True)
    clean(ax)
    ax.set_title("Score on our videos")
    ax.set_xlabel("Questions answered correctly (questions with a true answer)")
    best = summ["mean"].idxmax()
    n_q = int(summ["count"].max())
    save(fig, "own_1_overall.png", f"On our videos, {label(best)} scores highest: {100 * summ.loc[best, 'mean']:.0f}%",
         f"{n_q} questions with a true answer across {len(own_videos)} videos. With this few questions, one question ≈ {100 / n_q:.0f} points.")

    # Score per model and video.
    pv = own_scored.pivot_table(index="model", columns="video", values="score", aggfunc="mean").reindex(summ.index)
    nq = own_scored.drop_duplicates(["video", "question"]).groupby("video").size()
    fig, ax = plt.subplots(figsize=(1.9 * len(pv.columns) + 3.5, 0.6 * len(pv) + 1.6))
    heatmap(ax, (100 * pv.values).tolist(), [label(m) for m in pv.index], [f"{v}\n({nq.get(v, 0)} scored)" for v in pv.columns], lambda v: f"{v:.0f}%")
    ax.set_title("Score per model and video")
    hardest = pv.mean().idxmin()
    save(fig, "own_2_per_video.png", f"{hardest} is the hardest of our videos ({100 * pv.mean()[hardest]:.0f}% on average)")

    # Every question: right or wrong, with each model's answer.
    for video in own_videos:
        sub = own_scored[own_scored["video"] == video]
        if sub.empty:
            continue
        qs = sub.drop_duplicates("qid").sort_values("qid")
        ms = list(summ.index)
        fig, ax = plt.subplots(figsize=(6.5 + 1.25 * (len(ms) + 1), 0.6 * len(qs) + 1.5))
        ax.set_xlim(-0.5, len(ms) + 1.3)
        ax.set_ylim(len(qs) - 0.5, -0.5)
        for i, (_, q) in enumerate(qs.iterrows()):
            ax.add_patch(plt.Rectangle((-0.47, i - 0.45), 0.94, 0.9, color=NEUTRAL, lw=0))
            ax.text(0, i, textwrap.shorten(q["true_answer"], 12, placeholder="…"), ha="center", va="center", fontweight="bold")
            right = 0
            for j, m in enumerate(ms, start=1):
                hit = sub[(sub["qid"] == q["qid"]) & (sub["model"] == m)]
                if hit.empty:
                    ax.add_patch(plt.Rectangle((j - 0.47, i - 0.45), 0.94, 0.9, color=NEUTRAL, lw=0))
                    ax.text(j, i, "—", ha="center", va="center", color=INK_2)
                    continue
                s, ans = hit["score"].iloc[0], hit["model_answer"].iloc[0].strip()
                color, mark = (GOOD, "✓") if s == 1 else (BAD, "✗") if s == 0 else (PARTIAL, "~")
                right += s
                ax.add_patch(plt.Rectangle((j - 0.47, i - 0.45), 0.94, 0.9, color=color, lw=0))
                ax.text(j, i, f"{mark} {(first_word(ans) or '(empty)')[:9]}", ha="center", va="center", color="white", fontweight="bold")
            ax.text(len(ms) + 0.62, i, f"{right:g}/{len(ms)}", va="center", color=INK_2)
        ax.set_xticks(range(len(ms) + 1), ["True\nanswer"] + [label(m, newline=True) for m in ms], fontsize=9)
        ax.xaxis.tick_top()
        ax.set_yticks(range(len(qs)), [textwrap.shorten(f"Q{q['qid']}. {q['q']}", 72, placeholder="…") for _, q in qs.iterrows()])
        clean(ax)
        ax.set_title(f"{video}: each model's answer (right column: models correct)", pad=40)
        zero = int(sum(1 for qid in qs["qid"] if sub[sub["qid"] == qid]["score"].max() == 0))
        save(fig, f"own_3_questions_{os.path.splitext(video)[0]}.png",
             f"{video}: {len(qs)} questions" + (f", {zero} that no model answered correctly" if zero else ""),
             "✓ right   ✗ wrong   ~ partly right (numbers). Answers show the model's first word.")

    # Agreement between models.
    fw = own_scored.assign(a=own_scored["model_answer"].map(lambda a: first_word(a).lower())) \
                   .pivot_table(index=["video", "qid"], columns="model", values="a", aggfunc="first")
    ms = [m for m in summ.index if m in fw.columns]
    agree = [[np.nan if not (fw[a].notna() & fw[b].notna()).any()
              else 100 * (fw.loc[fw[a].notna() & fw[b].notna(), a] == fw.loc[fw[a].notna() & fw[b].notna(), b]).mean()
              for b in ms] for a in ms]
    fig, ax = plt.subplots(figsize=(1.1 * len(ms) + 3.5, 0.85 * len(ms) + 1.2))
    heatmap(ax, agree, [label(m) for m in ms], [label(m, newline=True) for m in ms], lambda v: f"{v:.0f}%")
    plt.setp(ax.get_xticklabels(), fontsize=9)
    ax.set_title("How often two models give the same answer (our videos)")
    off = pd.DataFrame(agree, index=ms, columns=ms).where(~np.eye(len(ms), dtype=bool))
    loner = off.mean().idxmin()
    save(fig, "own_4_agreement.png", f"{label(loner)} disagrees most with the others ({off.mean()[loner]:.0f}% agreement on average)")

# Open-ended answers (no true answer): length, empty, and runaway loops.
def looping(a):
    sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", a) if s.strip()]
    return len(sents) >= 6 and len(set(sents)) <= len(sents) / 2

open_q = own[(own["true_answer"].astype(str).str.strip() == "") & ~own["question"].str.contains("single word or phrase")]
for qtext, sub in open_q.groupby("q", sort=False):
    n = sub.pivot_table(index="model", columns="video", values="model_answer", aggfunc="first").reindex([m for m in MODELS if m in own_models])
    vals = [[len(a.split()) if isinstance(a, str) and a.strip() else np.nan for a in row] for row in n.values]
    fig, ax = plt.subplots(figsize=(1.9 * n.shape[1] + 3.5, 0.6 * len(n) + 1.6))
    vmax = max(150, np.nanmax(vals) if not np.all(np.isnan(vals)) else 150)
    im = heatmap(ax, vals, [label(m) for m in n.index], list(n.columns), lambda v: f"{v:.0f} words", vmax=vmax, empty="")
    empties, loops = 0, 0
    for i in range(n.shape[0]):
        for j in range(n.shape[1]):
            a = n.values[i, j]
            if isinstance(a, str) and not a.strip():
                ax.text(j, i, "empty", ha="center", va="center", color=BAD, fontweight="bold")
                empties += 1
            elif isinstance(a, str) and looping(a):
                ax.text(j, i + 0.3, "loops", ha="center", va="center", color="white" if len(a.split()) > vmax / 2 else BAD, fontsize=9)
                loops += 1
    ax.set_title(f"Answer length: “{textwrap.shorten(qtext, 60, placeholder='…')}”")
    note = []
    if empties:
        note.append(f"{empties} empty")
    if loops:
        note.append(f"{loops} stuck in a loop")
    save(fig, f"own_5_open_{re.sub('[^A-Za-z0-9]+', '_', qtext)[:30]}.png",
         f"Open-ended answers" + (f": {', '.join(note)}" if note else " vary in length"),
         "No true answer, so not scored. 'loops' = the same sentences repeated until the length limit.")

## Camera paths

In [ ]:
# @title 6. Camera-path maps
from PIL import Image as PILImage

def load_path(base, model, video):
    p = os.path.join(base, model, os.path.splitext(video)[0], "poses.npz")
    return np.load(p)["c2w"][:, :3, 3] if os.path.exists(p) else None

def draw_path(ax, pos):
    ax.set_xticks([])
    ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(True)
        s.set_color(GRID)
    if pos is None:
        ax.text(0.5, 0.5, "no poses", ha="center", va="center", color=INK_2, transform=ax.transAxes)
        return
    ax.plot(pos[:, 0], pos[:, 2], color=BLUE, lw=2)
    ax.plot(*pos[0, [0, 2]], "o", ms=8, mfc=SURFACE, mec=GOOD, mew=2)
    ax.plot(*pos[-1, [0, 2]], "X", ms=9, color=BAD, mec=SURFACE, mew=1)
    ax.set_aspect("equal", adjustable="datalim")

path_models_own = [m for m in own_models if any(load_path(RESULTS_DIR, m, v) is not None for v in own_videos)]
for video in own_videos:
    paths = {m: load_path(RESULTS_DIR, m, video) for m in path_models_own}
    if not any(p is not None for p in paths.values()):
        continue
    frames = next((os.path.join(RESULTS_DIR, m, os.path.splitext(video)[0], "frames.jpg") for m in own_models
                   if os.path.exists(os.path.join(RESULTS_DIR, m, os.path.splitext(video)[0], "frames.jpg"))), None)
    fig = plt.figure(figsize=(16, 7.2))
    grid = fig.add_gridspec(2, len(paths), height_ratios=[1.15, 1], hspace=0.25)
    if frames:
        axf = fig.add_subplot(grid[0, :])
        axf.imshow(PILImage.open(frames))
        axf.axis("off")
        axf.set_title("Some of the frames the model saw", fontsize=11, loc="left")
    for j, (m, pos) in enumerate(paths.items()):
        ax = fig.add_subplot(grid[1, j])
        draw_path(ax, pos)
        ax.set_title(label(m, newline=True), fontsize=10, loc="center")
    fig.suptitle(f"{video}: predicted camera path from above   (○ start  ✕ end, up = forward from the first frame)",
                 x=0.01, ha="left", fontweight="bold")
    save(fig, f"paths_own_{os.path.splitext(video)[0]}.png", f"{video}: where each model thinks the camera went",
         "Each panel has its own scale: the models' distances are relative, so compare shapes, turns and loops, not sizes.")

path_models_bench = [m for m in bench_models if any(load_path(BENCH_DIR, m, v) is not None for v in bench_videos)]
chunks = np.array_split(np.array(bench_videos, dtype=object), max(1, int(np.ceil(len(bench_videos) / 3)))) if path_models_bench else []
done = 0
for part, chunk in enumerate(chunks, start=1):
    chunk = list(chunk)
    fig, axes = plt.subplots(len(chunk), len(path_models_bench), figsize=(3.2 * len(path_models_bench) + 1.5, 3.0 * len(chunk)), squeeze=False)
    for i, v in enumerate(chunk):
        for j, m in enumerate(path_models_bench):
            draw_path(axes[i, j], load_path(BENCH_DIR, m, v))
            if i == 0:
                axes[i, j].set_title(label(m, newline=True), fontsize=11, loc="center")
            if j == 0:
                axes[i, j].set_ylabel(v.replace("scannet__", "").replace(".mp4", ""), fontsize=11, color=INK,
                                      rotation=0, ha="right", va="center", labelpad=10)
    fig.suptitle("VSI-Bench scenes: predicted camera paths from above (○ start  ✕ end)", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    save(fig, f"paths_bench_{part}.png", f"VSI-Bench camera paths, scenes {done + 1}–{done + len(chunk)} of {len(bench_videos)}",
         "Rows: scenes. Columns: models. Each panel has its own scale.")
    done += len(chunk)

## Slide deck

In [ ]:
# @title 7. Build the slide deck
from datetime import date
from pptx import Presentation
from pptx.dml.color import RGBColor
from pptx.util import Inches, Pt
from PIL import Image as PILImage

W, H = Inches(13.333), Inches(7.5)
prs = Presentation()
prs.slide_width, prs.slide_height = W, H
BLANK = prs.slide_layouts[6]

def rgb(hexstr):
    return RGBColor.from_string(hexstr.lstrip("#"))

def text(slide, x, y, w, h, s, size=18, bold=False, color=INK, align=None):
    tb = slide.shapes.add_textbox(x, y, w, h)
    tf = tb.text_frame
    tf.word_wrap = True
    for k, line in enumerate(s if isinstance(s, list) else [s]):
        p = tf.paragraphs[0] if k == 0 else tf.add_paragraph()
        p.text = line
        p.font.size, p.font.bold, p.font.color.rgb = Pt(size), bold, rgb(color)
        p.space_after = Pt(8)
        if align is not None:
            p.alignment = align
    return tb

def background(slide, color=SURFACE):
    fill = slide.background.fill
    fill.solid()
    fill.fore_color.rgb = rgb(color)

def title_slide(title, subtitle):
    s = prs.slides.add_slide(BLANK)
    background(s, "#0d366b")
    text(s, Inches(0.8), Inches(2.6), Inches(11.7), Inches(1.4), title, 40, True, "#ffffff")
    text(s, Inches(0.8), Inches(4.0), Inches(11.7), Inches(1.5), subtitle, 18, False, "#cde2fb")

def section_slide(title, lines):
    s = prs.slides.add_slide(BLANK)
    background(s)
    s.shapes.add_shape(1, 0, 0, Inches(0.25), H).fill.solid()
    s.shapes[-1].fill.fore_color.rgb = rgb(BLUE)
    s.shapes[-1].line.fill.background()
    text(s, Inches(0.8), Inches(0.7), Inches(11.7), Inches(1), title, 32, True)
    text(s, Inches(0.8), Inches(1.9), Inches(11.7), Inches(5), [f"•  {l}" for l in lines], 20, False, INK_2)

def chart_slide(path, title, note):
    s = prs.slides.add_slide(BLANK)
    background(s)
    text(s, Inches(0.5), Inches(0.3), Inches(12.3), Inches(0.9), title, 24, True)
    box_x, box_y, box_w, box_h = Inches(0.5), Inches(1.25), Inches(12.33), Inches(5.55 if note else 6.0)
    iw, ih = PILImage.open(path).size
    scale = min(box_w / iw, box_h / ih)
    w, h = int(iw * scale), int(ih * scale)
    s.shapes.add_picture(path, box_x + (box_w - w) // 2, box_y + (box_h - h) // 2, w, h)
    if note:
        text(s, Inches(0.5), Inches(6.9), Inches(12.3), Inches(0.5), note, 12, False, INK_2)

# Headline numbers, computed from the data above.
facts = []
if bench_table is not None and MAIN in bench_table.index and BASELINE in bench_table.index:
    oc = col("Overall")
    facts.append(f"VSI-Bench ({len(bench_videos)} ScanNet scenes, {n_total} questions): {MAIN} {bench_table.loc[MAIN, oc]:.1f}% vs "
                 f"{BASELINE} {bench_table.loc[BASELINE, oc]:.1f}%")
    pp, ps = f"{MAIN} (paper)", f"{BASELINE} (paper)"
    if pp in bench_table.index and ps in bench_table.index:
        facts.append(f"Paper, full benchmark: {bench_table.loc[pp, oc]:.1f}% vs {bench_table.loc[ps, oc]:.1f}%: "
                     f"our run shows a gain of {bench_table.loc[MAIN, oc] - bench_table.loc[BASELINE, oc]:+.1f} points "
                     f"vs the paper's {bench_table.loc[pp, oc] - bench_table.loc[ps, oc]:+.1f}")
if len(own_scored):
    facts.append(f"Our {len(own_videos)} videos ({int(summ['count'].max())} scored questions): best {label(best)} at "
                 f"{100 * summ.loc[best, 'mean']:.0f}%, {BASELINE} {100 * summ.loc[BASELINE, 'mean']:.0f}%"
                 if BASELINE in summ.index else f"Our videos: best {label(best)} at {100 * summ.loc[best, 'mean']:.0f}%")

title_slide(DECK_TITLE, [f"{len(MODELS)} models: " + ", ".join(m.replace("Cambrian-", "") for m in MODELS),
                         f"Generated {date.today():%B %d, %Y} from the results in Google Drive"])
section_slide("What we ran", [
    f"Five released Cambrian-P-7B checkpoints and the model they start from, {BASELINE} (no camera tokens)",
    f"VSI-Bench: {len(bench_videos)} ScanNet scenes, every question for those scenes, scored the VSI-Bench way" if len(bench) else "VSI-Bench: not found",
    f"Our videos: {len(own_videos)} iPhone videos ({', '.join(own_videos)}), our own multiple-choice questions with true answers",
    "Each Cambrian-P model also predicts where the camera is in every frame; we plot those paths",
])
section_slide("Headline numbers", facts or ["No scored results found"])
groups = [("VSI-Bench", "bench_"), ("Our videos", "own_"), ("Camera paths", "paths_")]
for name, prefix in groups:
    chosen = [c for c in CHARTS if os.path.basename(c[0]).startswith(prefix)]
    if not chosen:
        continue
    for path, title, note in chosen:
        chart_slide(path, title, note)
section_slide("How to read these results", [
    "Scores use VSI-Bench's rules: exact match on the first word for letters, partial credit for numbers (within 5% = full marks)",
    "Our videos have few questions, so one question moves a model's score by several points",
    "Camera paths are in each model's own relative units: compare shapes, not sizes",
    f"Full answer tables: own_videos_scored.csv and benchmark_scored.csv in {os.path.basename(REPORT_DIR)}/",
])
deck = os.path.join(REPORT_DIR, "cambrian_p_results.pptx")
prs.save(deck)
print(f"Saved {len(prs.slides)} slides to {deck}")
print("In Google Drive: right-click the file → Open with → Google Slides.")